In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import math

https://wuzzuf.net/search/jobs/?q=Machine%20Learning%20Scientist

In [15]:
# job = "Machine Learning Scientist"

In [17]:
# req  = requests.get('https://wuzzuf.net/search/jobs/?q=' + job.replace(' ', '%20'))
# req

<Response [200]>

In [44]:
# soup = BeautifulSoup(req.content,'lxml')
# soup

In [25]:
def find_no_of_jobs(job):
    req  = requests.get('https://wuzzuf.net/search/jobs/?q=' + job.replace(' ', '%20'))
    soup= BeautifulSoup(req.content,'lxml')
    jobs = int(soup.find({'strong'}).text.replace(',', ''))
    pages = math.ceil(jobs / 15)
    return jobs , pages


In [24]:
# find_no_of_jobs("Machine Learning Scientist")

12


In [35]:
def scrap_pages(query):

  num_jobs, num_pages = find_no_of_jobs(query)

  query = query.replace(' ', '%20')

  titles_lst, links_lst, occupations_lst, companies_lst , locations_list , specs_lst  = [], [], [], [], [] ,[]

  for pageNo in range(num_pages):
        page = requests.get('https://wuzzuf.net/search/jobs/?q=' + query + '&start=' + str(pageNo))
        soup = BeautifulSoup(page.content, 'lxml')

        titles = soup.find_all("h2", {'class': 'css-m604qf'})
        titles_lst += [title.a.text for title in titles]

        links_lst += [ title.a['href'] for title in titles]

        occupations = soup.find_all("div", {'class': 'css-1lh32fc'})
        occupations_lst += [occupation.text for occupation in occupations]

        companies = soup.find_all("a", {'class': 'css-17s97q8'})
        companies_lst += [company.text for company in companies]

        specs = soup.find_all("div", {'class': 'css-y4udm8'})
        specs_lst += [spec.text for spec in specs]

        locations = soup.find_all("span",{"class" : "css-5wys0k"})
        locations_list += [location.text for location in locations ]

  scraped_data = {}
  scraped_data['Title'] = titles_lst
  scraped_data['Link'] = links_lst
  scraped_data['Occupation'] = occupations_lst
  scraped_data['Company'] = companies_lst
  scraped_data['Specs'] = specs_lst
  scraped_data["Locations"] = locations_list

  df = pd.DataFrame(scraped_data)

  return scraped_data, df


In [37]:
# A , B = scrap_pages('Machine Learning Scientist')

In [39]:
def combine_dfs(dfs):
    df = pd.concat(dfs)
    df = df.drop_duplicates()
    return df

In [40]:
def combine_dicts(dicts):
  combined_dict = {}
  for key in dicts[0].keys():
        combined_dict[key] = []
        for dict in dicts:
            combined_dict[key] += dict[key]
  return combined_dict


#Scrap Ex_1

In [43]:
# import scrap_helper

data, df = scrap_pages('python developer')


df.to_csv('data_1.csv', index=False)
print("Save Successfuly")




Save Successfuly


# Scrap Ex_2

In [46]:
# import scrap_helper


da_dict, da_df = scrap_pages('data analysis')
ds_dict, ds_df = scrap_pages('data science')
bi_dict, bi_df = scrap_pages('business intelligence')


combined_dict = combine_dicts([ds_dict, da_dict, bi_dict])
combined_df = combine_dfs([ds_df, da_df, bi_df])





In [47]:
combined_df.to_csv('data_2.csv', index=False)